In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pymcel as pc 
from pymcel import constantes as const
import plotly.graph_objects as go

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


In [2]:
betas = np.linspace(0, 0.999, 100)
gammas = 1 / np.sqrt(1 - betas**2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=betas, y=gammas, mode='lines', name='Gamma'))
fig.update_layout(title='Factor de Lorentz (Gamma) vs Velocidad (Beta)', xaxis_title='Beta (v/c)', yaxis_title='Gamma (Factor de Lorentz)')
fig.show()

## Unidades 

las unidaes en EM son las unidades gaussianas, unidades electro magneticas absolutas.

$$
F = \frac{q_1 q_2}{4\pi\epsilon_0} 

$$



In [3]:
eps0 = const.eps0
UL = 100e3 #Km
UM = const.m_e
UT = UL/const.c

#* Uidades Derivadas

UV = UL / UT
UA = UL / UT**2
UF = UM * UA

#* Unidades Gaussianas 

UQ = UL * np.sqrt(4 * np.pi * eps0 * UF)
UE = UF / UQ
UB = UE / UV

C = 1

UL , UM , UT


(100000.0, np.float64(9.1093837015e-31), np.float64(0.00033356409519815205))

## Tensor de Faraday

In [4]:
def tensor_faraday(B0):

    Ex = Ey = Ez = 0
    Bx = By = 0
    Bz = B0

    F = np.array([
        [0,-Ex,-Ey,-Ez],
        [Ex,0,-Bz,By],
        [Ey,Bz,0,-Bx],
        [Ez,-By,Bx,0]
    ])

    return F

In [5]:
tensor_faraday(1)

array([[ 0,  0,  0,  0],
       [ 0,  0, -1,  0],
       [ 0,  1,  0,  0],
       [ 0,  0,  0,  0]])

## Ecuación de movimiento de una carga electrica

In [6]:
def edm_relativista(tau,Ys,m,q,B0):

    C = 1

    x0 , x1, x2 , x3 , U0 , U1 , U2 , U3 = Ys

    dx0_dtau = U0
    dx1_dtau = U1
    dx2_dtau = U2
    dx3_dtau = U3

    F = tensor_faraday(B0)
    Usub = np.array([U0,-U1,-U2,-U3])

    f = q/C * F @ Usub

    dU0_dtau = f[0] / m
    dU1_dtau = f[1] / m
    dU2_dtau = f[2] / m
    dU3_dtau = f[3] / m

    return np.array([dx0_dtau, dx1_dtau, dx2_dtau, dx3_dtau, dU0_dtau, dU1_dtau, dU2_dtau, dU3_dtau])



In [7]:
edm_relativista(0, np.array([0,0,0,0,0,1,1,1]), 1, 1, 1)

array([ 0.,  1.,  1.,  1.,  0.,  1., -1.,  0.])

## Condiciones Iniciales

In [8]:
# En coordenadas temporales y espaciales
t0 = 0
r0 = np.array([1.0, 0.0, 0.0])
v0 = np.array([0.0, 0.5, 0.2])

# En coordenadas espaciotemporales
x0 = np.array([C*t0, r0[0], r0[1], r0[2]])
gamma0 = 1 / np.sqrt(1 - np.linalg.norm(v0)**2 / C**2)
U0 = np.array([gamma0, 
               gamma0*v0[0], gamma0*v0[1], gamma0*v0[2]])

# Condiciones iniciales ahora si
Ys0 = np.concatenate((x0, U0))

In [9]:
from scipy.integrate import solve_ivp

m = 1
q = 1
B0 = 1
taus = np.linspace(0, 10, 1000)
solucion = solve_ivp(
    edm_relativista,
    (taus[0], taus[-1]),
    Ys0,
    t_eval=taus,
    args=(m, q, B0),
    method='Radau'
)

In [10]:
!pip install -Uq nbformat

In [11]:
ts = solucion.y[0] / C
xs = solucion.y[1]
ys = solucion.y[2]
zs = solucion.y[3]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode='lines',
            line=dict(width=4, color=ts, colorscale='Viridis'),
            name='Trayectoria'
        )
    ]
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data'
    ),
    template='plotly_white'
)

fig.show()